# LifeStack Training Notebook
### AI that handles life's worst Fridays
End-to-end training pipeline for the LifeStack simulation engine.

In [ ]:
!pip install groq openai chromadb sentence-transformers gradio matplotlib numpy pydantic openenv-core -q

In [ ]:
# Upload all LifeStack .py files
from google.colab import files
print('Upload all LifeStack .py files: life_state.py, reward.py, lifestack_env.py, simperson.py, conflict_generator.py, action_space.py, agent.py, memory.py, run_episode.py, train_trl.py')
uploaded = files.upload()
print(f'Uploaded: {list(uploaded.keys())}')

In [ ]:
import os
from google.colab import userdata
# Store your GROQ_API_KEY in Colab Secrets (key icon on left sidebar)
try:
    os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
    print('✅ API key loaded from Colab Secrets')
except:
    os.environ['GROQ_API_KEY'] = 'your_key_here'
    print('⚠️ Add your GROQ_API_KEY to Colab Secrets or paste it above')

In [ ]:
import sys
sys.path.append('.')
from core.life_state import LifeMetrics, ResourceBudget, DependencyGraph
from core.reward import compute_reward
from core.lifestack_env import LifeStackEnv
from intake.simperson import SimPerson
from agent.conflict_generator import generate_conflict, TaskGenerator
from agent.agent import LifeStackAgent
from agent.memory import LifeStackMemory

# Use TaskGenerator — gives a real task with routes, milestones, and events
_gen = TaskGenerator()
task = _gen.generate(domain='flight_crisis', difficulty=3)
conflict = generate_conflict(difficulty=3)  # for initial disruption

env = LifeStackEnv(task=task)
person = SimPerson()
print('\u2705 All modules loaded')
print(f'\u2705 Task: {task.goal} | Horizon: {task.horizon} steps | Routes: {len(task.viable_routes)} | Milestones: {len(task.milestones)}')
print(f'\u2705 Person: {person.get_personality_hint()}')

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())  # ensure project root is importable
from scripts.run_episode import run_episode
print('Running 3 sample episodes...\n')
rewards = []
for i, diff in enumerate([2, 3, 5], 1):
    result = run_episode(difficulty=diff, verbose=False)
    rewards.append(result['total_reward'])
    print(f'Episode {i} (difficulty {diff}): reward = {result["total_reward"]:.3f} | steps = {result["steps"]} | person = {result["person"]}')
print(f'\nAverage reward: {sum(rewards)/len(rewards):.3f}')

In [ ]:
!pip install "unsloth==2024.12.4" "trl>=0.9" "transformers>=4.45" peft accelerate datasets -q

!python train_trl.py

In [ ]:
from IPython.display import Image, display
import os

if os.path.exists('grpo_reward_curve.png'):
    display(Image('grpo_reward_curve.png'))
elif os.path.exists('trl_reward_curve.png'):
    display(Image('trl_reward_curve.png'))
else:
    print('Reward curve not found. Did training complete?')

In [ ]:
import os
import glob

checkpoints = glob.glob('lifestack_model/checkpoint-*')
print(f"Found {len(checkpoints)} checkpoints (saved every 50 steps).")
for ckpt in sorted(checkpoints):
    print(ckpt)

In [ ]:
from memory import LifeStackMemory
import shutil, os

print('=== BEFORE vs AFTER MEMORY ===\n')

# Without memory
if os.path.exists('./lifestack_memory'):
    shutil.move('./lifestack_memory', './lifestack_memory_backup')
result_no_mem = run_episode(difficulty=5, verbose=False)
print(f'Without memory | Reward: {result_no_mem["total_reward"]:.3f}')

# With memory
if os.path.exists('./lifestack_memory_backup'):
    shutil.move('./lifestack_memory_backup', './lifestack_memory')
result_with_mem = run_episode(difficulty=5, verbose=False)
print(f'With memory    | Reward: {result_with_mem["total_reward"]:.3f}')

improvement = result_with_mem['total_reward'] - result_no_mem['total_reward']
print(f'Improvement    : {improvement:+.3f}')
print(f'\nMemory stats: {LifeStackMemory().get_stats()}')

# Final Summary
**LifeStack:** Built an AI-driven sandbox for simulating complex life scenarios. It scales based on five fundamental personality traits and models resource budgets.
#### Cited Research:
- Generative Agents (Park et al., 2023)
- Large Language Models as Simulated Economic Agents (Horton, 2023)
- Evaluating LLMs for Social Scenarios (Li et al., 2023)
- Role-Playing in LLMs (Shanahan et al., 2023)

**Reward Improvement:** Evaluated baseline against retrieval-augmented dynamic memories.
**HuggingFace Demo:** Uploaded to HuggingFace Spaces.